# ORA-Q on Kaggle (T4)
Run top to bottom. Keep the GPU off until step 4 to save quota (Session options > Accelerator).

## 1. Install
transformers must be 5.x (the code uses its `q_proj/k_proj/v_proj/o_proj` names). Restart the session after this cell if Kaggle asks.

In [ ]:
!pip install -q -U "transformers>=5.0" h5py
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 2. Get the code
Option A (recommended): push the `ora-q` branch to your fork, then clone it.
Option B: upload the zip as a private Kaggle dataset, add it as input, and unzip it.

In [ ]:
# Option A
!git clone -q -b ora-q https://github.com/maiHydrogen/DA4022.git /kaggle/working/DA4022
# Option B
# !unzip -q /kaggle/input/<your-dataset>/DA4022-ora-q.zip -d /kaggle/working/
%cd /kaggle/working/DA4022
!ls Quantization_and_Pruning

## 3. Find the data
Add a NYU Depth V2 dataset that contains `nyu_depth_v2_labeled.mat` (Add Input > Datasets), or download it. `splits.mat` (Eigen split) is small; download it if your dataset does not have it. The links below are the commonly used NYU host; check them on the NYU page if they fail.

In [ ]:
!find /kaggle/input -name "*.mat" 2>/dev/null
# If missing (internet must be ON):
# !wget -q -O /kaggle/working/splits.mat http://horatio.cs.nyu.edu/mit/silberman/indoor_seg_sup/splits.mat
# !wget -q -O /kaggle/working/nyu_depth_v2_labeled.mat http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat

In [ ]:
MAT = "/kaggle/input/<dataset>/nyu_depth_v2_labeled.mat"   # edit after the find above
SPLITS = "/kaggle/working/splits.mat"                      # edit
import h5py, scipy.io
with h5py.File(MAT, 'r') as f: print(f['images'].shape, f['depths'].shape)   # (1449, 3, 640, 480) (1449, 640, 480)
s = scipy.io.loadmat(SPLITS); print(s['testNdxs'].shape, s['trainNdxs'].shape) # (654, 1) (795, 1)

## 4. Unit tests (CPU is fine)

In [ ]:
!python -m pytest -q Quantization_and_Pruning/tests/test_ora_q.py

## 5. Quick checks on 20 images (turn the GPU on now)
FP32 should be near 0.97. The 16-bit run must match FP32 closely; if not, something is wrong before any 4-bit result means anything.

In [ ]:
R = "Quantization_and_Pruning/experiments/ora_q/run_ora_q.py"
!python $R --mat $MAT --splits $SPLITS --methods fp32 --num-images 20 --no-latency --out /kaggle/working/quick.csv
!python $R --mat $MAT --splits $SPLITS --methods minmax --num-images 20 --act-bits 16 --weight-bits 16 --no-latency --out /kaggle/working/quick.csv

## 6. Full run, all 654 test images
Appends 4 rows to results.csv and writes role_report.json. Expect MinMax < Percentile < ORA-Q < FP32.

In [ ]:
!python $R --mat $MAT --splits $SPLITS --methods fp32 minmax percentile ora_q --num-images 654

In [ ]:
import pandas as pd, json
df = pd.read_csv('Quantization_and_Pruning/experiments/ora_q/results.csv')
print(df[['method','quality_value','abs_rel','latency_median_ms','peak_memory_mb']])
print(json.dumps(json.load(open('Quantization_and_Pruning/experiments/ora_q/role_report.json'))['operators'], indent=1))

## 7. Optional ablation: conv g = 16 (Table 2) instead of 8
Edit `conv: {g: 8 ...}` to 16 in `configs/ora_q/ora_q.yaml`, then rerun only ORA-Q (the role report is reused).

In [ ]:
# !sed -i 's/conv:    {g: 8/conv:    {g: 16/' Quantization_and_Pruning/configs/ora_q/ora_q.yaml
# !python $R --mat $MAT --splits $SPLITS --methods ora_q --num-images 654

## 8. Save
Download `results.csv` and `role_report.json` from `/kaggle/working/DA4022/Quantization_and_Pruning/experiments/ora_q/`, put them in your local clone, fill the paper card, commit, push, open the PR. Then stop the session.